In [1]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

def generate_synthetic_mto_data(num_records=5000):
    """
    Generates a realistic synthetic dataset for EPC Brownfield MTO Overrun Risk.
    Encodes engineering logic where extreme process conditions, complex metallurgy, 
    and poor site survey maturity drive material quantity growth.
    """
    
    # 1. Base Feature Generation
    projects = [f"PRJ-BF-{str(i).zfill(3)}" for i in range(1, 11)]
    project_id = np.random.choice(projects, num_records)
    
    line_tag = [f"PL-{str(i).zfill(5)}" for i in range(1, num_records + 1)]
    
    design_pressure = np.round(np.random.uniform(5.0, 250.0, num_records), 1)
    design_temp = np.round(np.random.uniform(-50.0, 550.0, num_records), 1)
    
    metallurgy = np.random.choice(
        ['CS', 'SS', 'Duplex', 'Inconel'], 
        num_records, 
        p=[0.60, 0.25, 0.10, 0.05] 
    )
    
    fluid_class = np.random.choice(
        ['Utility', 'Flammable', 'Toxic'], 
        num_records, 
        p=[0.40, 0.45, 0.15]
    )
    
    tie_in_density = np.random.poisson(lam=2.5, size=num_records)
    
    laser_scan_coverage = np.random.choice(
        [0.0, np.random.uniform(30, 80), np.random.uniform(80, 100)], 
        num_records, 
        p=[0.20, 0.30, 0.50]
    )
    laser_scan_coverage = np.where(laser_scan_coverage > 100, 100, laser_scan_coverage)
    
    feed_gate_maturity = np.random.choice(['30%', '60%', '90%'], num_records)
    
    feed_mto_qty = np.round(np.random.lognormal(mean=4.0, sigma=1.0, size=num_records), 1)
    feed_mto_qty = np.maximum(feed_mto_qty, 10.0)

    # 2. Engineering Logic: Calculate MTO Growth Multipliers
    growth_multiplier = np.random.uniform(1.02, 1.05, num_records)
    
    growth_multiplier += np.where(laser_scan_coverage == 0.0, np.random.uniform(0.10, 0.20, num_records), 0)
    growth_multiplier += np.where((laser_scan_coverage > 0) & (laser_scan_coverage < 50), np.random.uniform(0.05, 0.10, num_records), 0)
    
    growth_multiplier += (tie_in_density * np.random.uniform(0.01, 0.03, num_records))
    
    # CORRECTED LINE HERE
    growth_multiplier += np.where(np.isin(metallurgy, ['Duplex', 'Inconel']), np.random.uniform(0.03, 0.08, num_records), 0)
    
    growth_multiplier += np.where(design_pressure > 150.0, np.random.uniform(0.02, 0.06, num_records), 0)
    growth_multiplier += np.where(fluid_class == 'Toxic', np.random.uniform(0.02, 0.05, num_records), 0)
    
    growth_multiplier += np.where(feed_gate_maturity == '30%', np.random.uniform(0.05, 0.10, num_records), 0)
    growth_multiplier -= np.where(feed_gate_maturity == '90%', np.random.uniform(0.01, 0.03, num_records), 0)
    
    noise = np.random.normal(0, 0.02, num_records)
    growth_multiplier += noise
    
    growth_multiplier = np.maximum(growth_multiplier, 0.98)
    
    # 3. Calculate Final Targets
    final_asbuilt_qty = np.round(feed_mto_qty * growth_multiplier, 1)
    
    mto_growth_pct = np.round(((final_asbuilt_qty - feed_mto_qty) / feed_mto_qty) * 100, 2)
    high_risk_flag = np.where(mto_growth_pct > 15.0, 1, 0)
    
    # 4. Compile the DataFrame
    df = pd.DataFrame({
        'Project_ID': project_id,
        'Line_Tag': line_tag,
        'Design_Pressure_barg': design_pressure,
        'Design_Temp_C': design_temp,
        'Metallurgy': metallurgy,
        'Fluid_Class': fluid_class,
        'Tie_In_Density': tie_in_density,
        'Laser_Scan_Coverage_Pct': np.round(laser_scan_coverage, 1),
        'FEED_Gate_Maturity': feed_gate_maturity,
        'FEED_MTO_Qty': feed_mto_qty,
        'Final_AsBuilt_Qty': final_asbuilt_qty,
        'MTO_Growth_Pct': mto_growth_pct,
        'High_Risk_Flag': high_risk_flag
    })
    
    return df

if __name__ == "__main__":
    print("Generating synthetic brownfield MTO dataset...")
    mto_dataset = generate_synthetic_mto_data(num_records=8500)
    
    # Save directly to your specific absolute local path using a raw string (r"...")
    save_path = r"C:\Users\Jam Villanueva\Documents\AIM PGD AIML\1_Capstone Project\data\synthetic_brownfield_mto_data.csv"
    mto_dataset.to_csv(save_path, index=False)
    
    print(f"Dataset generated successfully with {mto_dataset.shape[0]} records.")
    print(f"Saved to: {save_path}")
    print(f"High-Risk Overruns (>15%): {mto_dataset['High_Risk_Flag'].sum()} records ({(mto_dataset['High_Risk_Flag'].mean() * 100):.1f}% of total).")
    
    print("\nFirst 5 records:")
    print(mto_dataset.head())

Generating synthetic brownfield MTO dataset...
Dataset generated successfully with 8500 records.
Saved to: C:\Users\Jam Villanueva\Documents\AIM PGD AIML\1_Capstone Project\data\synthetic_brownfield_mto_data.csv
High-Risk Overruns (>15%): 5320 records (62.6% of total).

First 5 records:
   Project_ID  Line_Tag  Design_Pressure_barg  Design_Temp_C Metallurgy  \
0  PRJ-BF-007  PL-00001                  33.0          145.9         SS   
1  PRJ-BF-004  PL-00002                 132.0          299.6         CS   
2  PRJ-BF-008  PL-00003                 148.9          140.2         CS   
3  PRJ-BF-005  PL-00004                  47.7          316.4         CS   
4  PRJ-BF-007  PL-00005                 240.9          530.1         SS   

  Fluid_Class  Tie_In_Density  Laser_Scan_Coverage_Pct FEED_Gate_Maturity  \
0   Flammable               1                     97.3                30%   
1       Toxic               2                     42.0                90%   
2     Utility               3 

In [2]:
import pandas as pd

# Load the dataset generated in the previous step
df = pd.read_csv('synthetic_brownfield_mto_data.csv')

# Display feature types and confirm zero unexpected nulls
print("--- Data Info ---")
df.info()

# Summarize distributions to identify outliers (e.g., max values in MTO_Growth_Pct)
print("\n--- Statistical Summary (Outlier Detection) ---")
display(df.describe())

# Show category breakdowns
print("\n--- Categorical Feature Distributions ---")
for col in ['Metallurgy', 'Fluid_Class', 'FEED_Gate_Maturity']:
    print(f"\n{df[col].value_counts(normalize=True) * 100}")

--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8500 entries, 0 to 8499
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Project_ID               8500 non-null   object 
 1   Line_Tag                 8500 non-null   object 
 2   Design_Pressure_barg     8500 non-null   float64
 3   Design_Temp_C            8500 non-null   float64
 4   Metallurgy               8500 non-null   object 
 5   Fluid_Class              8500 non-null   object 
 6   Tie_In_Density           8500 non-null   int64  
 7   Laser_Scan_Coverage_Pct  8500 non-null   float64
 8   FEED_Gate_Maturity       8500 non-null   object 
 9   FEED_MTO_Qty             8500 non-null   float64
 10  Final_AsBuilt_Qty        8500 non-null   float64
 11  MTO_Growth_Pct           8500 non-null   float64
 12  High_Risk_Flag           8500 non-null   int64  
dtypes: float64(6), int64(2), object(5)
memory usage: 863.4+ KB



,Design_Pressure_barg,Design_Temp_C,Tie_In_Density,Laser_Scan_Coverage_Pct,FEED_MTO_Qty,Final_AsBuilt_Qty,MTO_Growth_Pct,High_Risk_Flag
count,8500.000000,8500.000000,8500.000000,8500.000000,8500.000000,8500.000000,8500.000000,8500.000000
mean,127.964447,248.880118,2.525529,60.477776,90.031788,107.139012,18.808238,0.625882
std,70.561091,172.870729,1.602719,38.993438,115.310361,139.296279,9.285599,0.483923
min,5.000000,-50.000000,0.000000,0.000000,10.000000,9.900000,-2.130000,0.000000
25%,67.900000,98.675000,1.000000,42.000000,27.600000,32.300000,11.920000,0.000000
50%,128.600000,246.650000,2.000000,42.000000,54.700000,64.800000,18.000000,1.000000
75%,188.625000,397.500000,3.250000,97.300000,106.800000,127.025000,25.000000,1.000000
max,250.000000,549.900000,12.000000,97.300000,2272.900000,2924.700000,56.410000,1.000000



--- Categorical Feature Distributions ---

Metallurgy
CS         59.117647
SS         26.282353
Duplex      9.764706
Inconel     4.835294
Name: proportion, dtype: float64

Fluid_Class
Flammable    44.282353
Utility      40.776471
Toxic        14.941176
Name: proportion, dtype: float64

FEED_Gate_Maturity
60%    33.552941
30%    33.517647
90%    32.929412
Name: proportion, dtype: float64


In [3]:
import os
print(os.getcwd())

C:\Users\Jam Villanueva
